# Milestone 3 — Reference Matching Pipeline & Full Benchmarking

Notebook thực thi toàn bộ quy trình **Milestone 3** theo đặc tả [matching.md](../Project_description/matching.md) và [benchmark_m3.md](../Project_description/benchmark_m3.md):
1. **Kết nối Google Drive & Sao chép dữ liệu Parquet về SSD cục bộ** (`/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data/parquet/` $\rightarrow$ `/content/local_parquet/`).
2. **Data Cleaning & Labelling:** Gán nhãn thủ công ($\ge 5$ bài báo, mỗi bài $\ge 20$ mục) và gán nhãn tự động ($\ge 10\%$ số bài ngoài tập thủ công).
3. **Feature Engineering:** Xây dựng 16 đặc trưng cặp `(bibtex_entry, reference)` (bao gồm ngữ cảnh trích dẫn từ cây `hierarchy` của Milestone 2).
4. **Huấn luyện & Benchmark 4 Mô hình:**
   - Baseline 1: **TF-IDF Cosine thuần**
   - Baseline 2: **Logistic Regression**
   - Model 3: **Random Forest**
   - Model 4: **LightGBM**
5. **Đo Benchmark Tài nguyên Hệ thống (`m3_system_benchmark.csv`):** Đo thời gian từng pha, Peak/Avg RAM, Peak Disk và dung lượng thành phẩm bằng `SystemBenchmarkTracker`.
6. **Xuất `predictions.parquet`** từ mô hình tốt nhất (theo `Valid MRR@5`) và đồng bộ kết quả về Google Drive.


In [ ]:
#@title 1. Kết nối Google Drive, lấy mã nguồn & chuẩn bị thư mục làm việc { display-mode: "form" }
import os, sys, subprocess
from pathlib import Path

BRANCH = "feat/m3-matching"
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/arxiv-code")
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/arXiv-Project/data")
    DRIVE_PARQUET_DIR = DRIVE_DATA_DIR / "parquet"
    DRIVE_CSV_DIR = DRIVE_DATA_DIR / "csv"

    if not (CODE_DIR / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "https://github.com/ahtuan171/arXiv-Project.git", str(CODE_DIR),
        ], check=True)
    else:
        actual = subprocess.check_output(
            ["git", "-C", str(CODE_DIR), "branch", "--show-current"], text=True
        ).strip()
        if actual != BRANCH:
            subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "origin", BRANCH], check=True)
            subprocess.run(["git", "-C", str(CODE_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "pull", "origin", BRANCH], check=True)

    LOCAL_PARQUET_DIR = Path("/content/local_parquet")
    LOCAL_OUT_PARQUET_DIR = Path("/content/local_out/parquet")
    LOCAL_OUT_CSV_DIR = Path("/content/local_out/csv")
else:
    CODE_DIR = Path.cwd().resolve()
    if not (CODE_DIR / "src").is_dir() and (CODE_DIR.parent / "src").is_dir():
        CODE_DIR = CODE_DIR.parent
    DRIVE_PARQUET_DIR = None
    DRIVE_CSV_DIR = None
    LOCAL_PARQUET_DIR = CODE_DIR / "data" / "parquet"
    LOCAL_OUT_PARQUET_DIR = LOCAL_PARQUET_DIR
    LOCAL_OUT_CSV_DIR = CODE_DIR / "data" / "csv"

os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("Môi trường      :", "Google Colab" if IN_COLAB else "Local")
print("Mã nguồn        :", CODE_DIR)
print("Parquet làm việc:", LOCAL_PARQUET_DIR)
if DRIVE_PARQUET_DIR:
    print("Parquet trên Drive:", DRIVE_PARQUET_DIR)


In [ ]:
#@title 2. Cài đặt thư viện (LightGBM, Scikit-learn, PyArrow) & Copy Parquet từ Drive về SSD cục bộ
import shutil
import subprocess, sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "scikit-learn", "pyarrow", "pandas", "psutil"], check=True)
    LOCAL_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_OUT_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_OUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

    if not DRIVE_PARQUET_DIR.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục Parquet trên Drive: {DRIVE_PARQUET_DIR}")

    for fname in ["papers.parquet", "references.parquet", "elements.parquet", "hierarchy.parquet", "bibtex.parquet"]:
        src = DRIVE_PARQUET_DIR / fname
        dst = LOCAL_PARQUET_DIR / fname
        if src.is_file():
            if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
                print(f"Đang copy {fname} ({src.stat().st_size / (1024*1024):.1f} MB) về SSD cục bộ...", flush=True)
                shutil.copy2(src, dst)
            else:
                print(f"Đã có sẵn trên SSD: {fname}")
        else:
            print(f"⚠️ Lưu ý: Không tìm thấy {src}")

    # Copy sẵn m3_manual_labels.csv nếu nhóm đã chuẩn bị trên Drive
    if DRIVE_CSV_DIR and (DRIVE_CSV_DIR / "m3_manual_labels.csv").is_file():
        shutil.copy2(DRIVE_CSV_DIR / "m3_manual_labels.csv", LOCAL_OUT_CSV_DIR / "m3_manual_labels.csv")
        print("Đã nạp m3_manual_labels.csv từ Drive.")
else:
    print("Chạy ở local, sử dụng trực tiếp file trong:", LOCAL_PARQUET_DIR)


In [ ]:
#@title 3. Cấu hình & Chạy Pipeline Milestone 3 (Huấn luyện 4 Mô hình + Benchmark Hệ thống) { display-mode: "form" }
AUTO_FRACTION = 0.12  #@param {type:"number"}
RANDOM_SEED = 42  #@param {type:"integer"}
BUILD_HF_DATASET = False  #@param {type:"boolean"}

from src.scripts.run_m3_pipeline import run_m3_pipeline

hf_dir = (LOCAL_OUT_PARQUET_DIR.parent / "hf_dataset") if BUILD_HF_DATASET else None
summary = run_m3_pipeline(
    parquet_dir=LOCAL_PARQUET_DIR,
    out_parquet_dir=LOCAL_OUT_PARQUET_DIR,
    csv_dir=LOCAL_OUT_CSV_DIR,
    auto_fraction=AUTO_FRACTION,
    seed=RANDOM_SEED,
    hf_dir=hf_dir,
)
print(f"\n✅ Hoàn tất Milestone 3 trong {summary['elapsed_sec']}s! Mô hình tốt nhất: {summary['model']['selected_model']}")


In [ ]:
#@title 4. Kết quả Benchmark 4 Mô hình, Tài nguyên Hệ thống & Feature Importance
import pandas as pd
from IPython.display import display

print("=== 1. THỐNG KÊ GÁN NHÃN (DATA LABELLING) ===")
display(pd.DataFrame([summary["labelling"]]))

print("\n=== 2. BẢNG SO SÁNH 4 MÔ HÌNH (MODEL & RANKING BENCHMARK) ===")
model_bench_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_model_benchmark.csv")
display(model_bench_df)
print("🏆 Mô hình được chọn xuất predictions.parquet:", summary["model"]["selected_model"])

print("\n=== 3. BENCHMARK TÀI NGUYÊN HỆ THỐNG (m3_system_benchmark.csv) ===")
sys_bench_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_system_benchmark.csv")
display(sys_bench_df)

print("\n=== 4. ĐỘ QUAN TRỌNG ĐẶC TRƯNG (PERMUTATION IMPORTANCE) ===")
feat_df = pd.read_csv(LOCAL_OUT_CSV_DIR / "m3_feature_importance.csv")
display(feat_df)


In [ ]:
#@title 5. Kiểm tra Schema predictions.parquet & Đồng bộ kết quả lên Google Drive
import shutil
import pyarrow.parquet as pq

pred_path = LOCAL_OUT_PARQUET_DIR / "predictions.parquet"
print("PyArrow Schema của predictions.parquet:")
print(pq.read_schema(pred_path))

df_pred = pd.read_parquet(pred_path)
print(f"\nTổng số dòng: {len(df_pred):,}")
print("Phân bố theo partition:")
print(df_pred["partition"].value_counts())
display(df_pred[df_pred["partition"] == "test"].head(15))

if IN_COLAB and DRIVE_PARQUET_DIR and DRIVE_CSV_DIR:
    DRIVE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_CSV_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pred_path, DRIVE_PARQUET_DIR / "predictions.parquet")
    print(f"\n✅ Đã xuất predictions.parquet lên Drive: {DRIVE_PARQUET_DIR / 'predictions.parquet'}")
    for fname in [
        "m3_model_benchmark.csv",
        "m3_system_benchmark.csv",
        "m3_feature_importance.csv",
        "m3_labelling_stats.csv",
        "m3_manual_labels.csv",
        "m3_evaluation_summary.json",
        "m3_best_model.pkl",
    ]:
        src_f = LOCAL_OUT_CSV_DIR / fname
        if src_f.is_file():
            shutil.copy2(src_f, DRIVE_CSV_DIR / fname)
    print(f"✅ Đã đồng bộ toàn bộ file benchmark & trọng số mô hình lên Drive: {DRIVE_CSV_DIR}")
